# Usage | 3. Existing bike network
This notebook explains how GrowBikeNet can extend an existing bike network.

**Parameters covered**: `existing_network_spacing`  
**Modules covered**: `bnl.osm`

We start every Usage notebook with the standard way of importing GrowBikeNet:

In [5]:
import growbikenet as gbn

So far GrowBikeNet was executed with the default parameter setting `existing_network_spacing=None`, which instructed GrowBikeNet to ignore existing bicycle infrastructure. This works for most cities, as existing infrastructure is usually negligible and one might as well just start from scratch. However, there are some cities with an already existing substantial network which would be useful to incorporate into the growth process.

## Exploring the existing bicycle network

Let us consider Athens, Greece. Before working with GrowBikeNet, you might want to explore a city's existing bicycle network. This can be done with the `osm` module of BikeNetLib - BikeNetKit's core utilities package. Its `PBI_CUSTOM_FILTER` is a filter that downloads the existing bicycle network considered by GrowBikeNet. The parameter `retain_all=True` ensures that all (disconnected) components of the network are downloaded, not just the largest one:

In [6]:
import osmnx as ox
import bikenetlib as bnl

g = ox.graph_from_place("Municipality of Athens", 
    custom_filter=bnl.osm.PBI_CUSTOM_FILTER, retain_all=True)
edges = ox.graph_to_gdfs(g, nodes=False, edges=True)

edges.explore(
    tiles="https://tiles.stadiamaps.com/tiles/alidade_smooth/{z}/{x}/{y}{r}.png", 
    attr="(C) Stadia Maps (C) OpenMapTiles (C) OpenStreetMap contributors",
    style_kwds={"weight": 2, "color": "#9999cc"},
)

Exploration shows that this network is highly fragmented and covers only a small part of the city - the typical issue with bicycle networks and a reason why tools like GrowBikeNet are needed.

## Adding seed points on the existing bike network

By calling GrowBikeNet with the parameter `existing_network_spacing='auto'` or with a positive integer, it will work with this existing bicycle network.

In this case, the process of generating seed points is amended beforehand:
- Consider all network components of the existing bike network that have a minimum length. This ensures that tiny, insignificant pieces are ignored.
- On these components, choose a random first seed point.
- Choose the closest seed point on the components that is at least `existing_network_spacing` meters away. The `'auto'` option automatically chooses a recommended distance, at 50% of the `seed_point_grid_spacing`.
- Proceed with the previous step until no more seed points can be placed on the components.
- Now generate all the other seed points as usual, but do not consider seed points that are too close to already existing seed points.

Let us run GrowBikeNet on Athens with the `existing_network_spacing='auto'` option and observe the results:

In [7]:
edges_ordered = gbn.growbikenet("Municipality of Athens",
    existing_network_spacing='auto',)

RUNNING GROWBIKENET FOR CITY: Municipality of Athens
betweenness | auto | from existing bike network 
----------------------------------------------╮


Rerouting              : 100%|████████████████| 66/66 [00:01<00:00, 46.89edge/s]
/Users/mszell/Github/BikeNetKit/GrowBikeNet/.pixi/envs/default/lib/python3.15/site-packages/pandas/core/generic.py:6448: FutureWarning: Overriding the CRS of a GeoDataFrame that already has CRS. This unsafe behavior will be deprecated in future versions. Use GeoDataFrame.set_crs method instead
  return object.__setattr__(self, name, value)
Exporting data         : 100%|████████████████| 4/4 [00:00<00:00, 29.42file/s]

----------------------------------------------╯
Data exported to ./results/
----------------------------------------------
FINISHED IN 0:00:07


The existing bike network is saved as multilinestring into the first row of the resulting geodataframe with several entries being `None`:

In [8]:
edges_ordered.head()

,betweenness,geometry,source,target,ordering,length,length_cumulative
0,NaN,"MULTILINESTRING ((23.72545 37.97514, 23.72538 ...",NaN,NaN,0,35796,35796
1,0.144615,"MULTILINESTRING ((23.72878 37.98723, 23.72869 ...",2.511366e+08,9.566345e+07,1,1266,37063
2,0.141538,"MULTILINESTRING ((23.72819 38.00077, 23.72868 ...",5.495258e+08,3.584769e+08,2,687,37750
3,0.138462,"LINESTRING (23.73014 37.97359, 23.73093 37.973...",6.707880e+09,7.229807e+09,3,496,38247
4,0.138462,"LINESTRING (23.72878 37.98723, 23.72889 37.987...",5.495258e+08,9.566345e+07,4,1595,39842


To visualize the outcome, we plot first the existing bike network (first row) in blue, then the grown network (all other rows) in green. To add layer control in the top right of the map, we import folium:

In [9]:
import folium

viz = edges_ordered.iloc[:1].explore(
    tiles="https://tiles.stadiamaps.com/tiles/alidade_smooth/{z}/{x}/{y}{r}.png", 
    attr="(C) Stadia Maps (C) OpenMapTiles (C) OpenStreetMap contributors",
    style_kwds={"weight": 2, "color": "#9999cc"},
    name="Existing bike network",
)
viz = edges_ordered.iloc[1:].explore(
    m=viz, 
    style_kwds={"weight": 3, "color": "#096a51"},
    name="Grown bike network",
)
folium.LayerControl().add_to(viz)
viz

Note how the short existing pieces in the northeast are ignored, but the other big enough components are incorporated into the growth process.

## Comparing with growth from scratch

Let us add the outcome from growth from scratch (without the existing network) in orange to see the difference:

In [10]:
edges_ordered_from_scratch = gbn.growbikenet("Municipality of Athens")

RUNNING GROWBIKENET FOR CITY: Municipality of Athens
betweenness | auto | from scratch
----------------------------------------------╮


Exporting data         : 100%|████████████████| 3/3 [00:00<00:00, 141.39file/s]

----------------------------------------------╯
Data exported to ./results/
----------------------------------------------
FINISHED IN 0:00:05


In [11]:
viz = edges_ordered.iloc[:1].explore(
    tiles="https://tiles.stadiamaps.com/tiles/alidade_smooth/{z}/{x}/{y}{r}.png", 
    attr="(C) Stadia Maps (C) OpenMapTiles (C) OpenStreetMap contributors",
    style_kwds={"weight": 2, "color": "#9999cc"},
    name="Existing bike network",
)
viz = edges_ordered_from_scratch.explore(
    m=viz, 
    style_kwds={"weight": 6, "color": "#f19730"},
    name="Grown bike network (from scratch)",
)
viz = edges_ordered.iloc[1:].explore(
    m=viz, 
    style_kwds={"weight": 3, "color": "#096a51"},
    name="Grown bike network (with existing network)",
)
folium.LayerControl().add_to(viz)
viz

In general, the network which accounts for existing infrastructure will be longer than the one grown from scratch, in this case

In [12]:
int((edges_ordered.iloc[-1].length_cumulative-
    edges_ordered.iloc[0].length_cumulative)/1000)

55

kilometers compared to

In [13]:
int((edges_ordered_from_scratch.iloc[-1].length_cumulative)/1000)

43

kilometers, as seed points are generated more densely.